# MLB win-probability model

Trains the project's **sport-agnostic** Elo + feature pipeline (`walk_forward`, the same one the NBA model uses) on **38k real MLB games** (2010–2026, `data/mlb.duckdb`, from the free MLB Stats API). No new model code — just baseball data through the existing machinery. Out-of-sample (fit earlier games, score later).

**Expect modest skill.** Baseball is high-variance and team-Elo ignores the single biggest factor — the *starting pitcher*. A good pregame team-only model tops out around 56–58% accuracy; that's the honest ceiling here, not a bug.

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts'))
from sportsball.pipelines._elo import walk_forward
from sportsball.quant import features as feat
from train_eval_duckdb import _pipeline
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
K, HFA = 4, 24   # baseball Elo: low K, low home edge (538-style)

In [ ]:
con = duckdb.connect(str(ROOT/'data'/'mlb.duckdb'), read_only=True)
raw = con.execute('SELECT game_date,home_team,away_team,home_score,away_score FROM games ORDER BY game_date').fetchall()
con.close()
results = [(d,h,a,hs,as_) for (d,h,a,hs,as_) in raw]
frows, snaps = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=120, form_window=15)
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
elo_p = np.array([r.exp_home for r in frows])
cut = int(0.85*len(X))
base = y[cut:].mean()
print(f'{len(X)} games | test {len(y)-cut} | home-win base rate {base:.3f}')

## How much skill? (out-of-sample holdout)

Three reference points: always-pick-home (the base rate), the raw **Elo** probability, and a **logistic on the full 9-feature vector** (Elo + run-differential + rest/form; roster/availability/market features are inert for MLB — no data yet).

In [ ]:
def report(name, p):
    return {'model':name,'accuracy':round(accuracy_score(y[cut:],(p[cut:]>=0.5).astype(int)),4),
            'log_loss':round(log_loss(y[cut:],p[cut:],labels=[0,1]),4),
            'brier':round(brier_score_loss(y[cut:],p[cut:]),4)}
# always-home baseline as a constant prob = train base rate
home_const = np.full(len(y), y[:cut].mean())
logit = _pipeline().fit(X[:cut], y[:cut]); full_p = np.empty(len(y)); full_p[:] = np.nan
full_p[cut:] = logit.predict_proba(X[cut:])[:,1]; full_p[:cut]=logit.predict_proba(X[:cut])[:,1]
rows=[report('always-home (base rate)', home_const), report('Elo only', elo_p), report('full-feature logistic', full_p)]
pd.DataFrame(rows)

## Calibration

Does a 55%-predicted game actually win ~55%? (Narrow range — baseball probabilities cluster near 0.5.)

In [ ]:
from sklearn.calibration import calibration_curve
fp, mp = calibration_curve(y[cut:], full_p[cut:], n_bins=8, strategy='quantile')
plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'--',color='gray',label='perfect')
plt.plot(mp, fp, 'o-', label='MLB model'); plt.xlabel('predicted P(home win)'); plt.ylabel('observed')
plt.title('MLB model calibration (holdout)'); plt.legend(); plt.show()

## Team strength (final Elo ratings)

Where the model has each team after the last game ingested — a sanity check that it learned real baseball.

In [ ]:
rate = pd.DataFrame([(t, s.elo) for t,s in snaps.items()], columns=['team','elo']).sort_values('elo', ascending=False)
fig, ax = plt.subplots(1,2, figsize=(13,6))
sns.barplot(data=rate.head(10), y='team', x='elo', ax=ax[0], color='seagreen'); ax[0].set_title('Top 10 by Elo')
sns.barplot(data=rate.tail(10), y='team', x='elo', ax=ax[1], color='indianred'); ax[1].set_title('Bottom 10 by Elo')
for a in ax: a.set_xlim(rate.elo.min()-20, rate.elo.max()+20)
plt.tight_layout(); plt.show()

## Skill over time

Rolling accuracy of the model vs always-picking-home, across the test window.

In [ ]:
win=400
correct = (full_p[cut:]>=0.5).astype(int)==y[cut:]
home_correct = y[cut:]==1
roll_m = pd.Series(correct).rolling(win, min_periods=50).mean()
roll_h = pd.Series(home_correct).rolling(win, min_periods=50).mean()
plt.figure(figsize=(11,5))
plt.plot(roll_m.values, label='model', color='seagreen')
plt.plot(roll_h.values, label='always-home', color='gray', ls='--')
plt.axhline(0.5, color='crimson', lw=.6, label='coin flip')
plt.ylabel(f'rolling accuracy (window {win})'); plt.xlabel('test games'); plt.legend()
plt.title('MLB model skill over time'); plt.show()

## Verdict

The model has **real but modest** skill — a few points above always-picking-home, log-loss meaningfully below a coin flip, and reasonable calibration. The Elo ratings recover plausible team strength.

**The ceiling is the starting pitcher.** Team-level Elo can't see that tonight's game is Ace vs. swingman — the dominant lever in MLB. That's the obvious next feature (starting-pitcher rating / recent form), the baseball analog of NBA injuries but more decisive. Park factors, bullpen state, and travel are smaller follow-ons. Market odds (`market_logit`) would also activate once MLB closing lines are ingested.